# Data Normalization v2: normalise SOURCE only

⚠️ **Important:** TARGET's Y is a binary classification label (0-1), so it is not normalised

- SOURCE: Log10 + Z-score (continuous values)
- TARGET: Y used as is (0-1 labels)

In [17]:
import pandas as pd
import numpy as np
import os
from rdkit import Chem

source_deduplicated_dir = r'../data/in_vitro_data_deduplicated'
pksmarts_dir = r'../data/pksmarts_data'
output_dir = r'../data/normalized_data'
os.makedirs(output_dir, exist_ok=True)

print("Data Normalization v2: SOURCE only\n")

Data Normalization v2: SOURCE only



## SOURCE normalisation (Log10 + Z-score)

In [18]:
source_files = [
    ('clearance', ['chembl_CL_raw_smart_deduplicated.csv', 'biogen_CL_raw_smart_deduplicated.csv']),
    ('half_life', ['tdc_half_life_smart_deduplicated.csv']),
    ('fu', ['chembl_Fu_raw_smart_deduplicated.csv']),
]

def get_value_column(df):
    """Find the value column in a DataFrame"""
    if 'value' in df.columns:
        return 'value'
    elif 'Y' in df.columns:
        return 'Y'
    else:
        return None

def log10_transform(df, value_col):
    """Log10 transform"""
    df = df.copy()
    negative_count = (df[value_col] <= 0).sum()
    if negative_count > 0:
        print(f"    Removing {negative_count} non-positive values")
        df = df[df[value_col] > 0].reset_index(drop=True)
    df[value_col] = np.log10(df[value_col])
    return df

def zscore_normalize(df, value_col):
    """Z-score normalisation"""
    df = df.copy()
    mean = df[value_col].mean()
    std = df[value_col].std()
    df[value_col] = (df[value_col] - mean) / std if std > 0 else 0
    return df, mean, std

print("Normalizing SOURCE data (Log10 + Z-score)...\n")

normalized_sources = {}
norm_stats = {}

for prop_name, files in source_files:
    print(f"Property: {prop_name}")
    
    source_dfs = []
    stats = []
    
    for file in files:
        file_path = os.path.join(source_deduplicated_dir, file)
        if not os.path.exists(file_path):
            print(f"  ⚠️ {file} not found, skipping")
            continue
        
        df = pd.read_csv(file_path)
        original_rows = len(df)
        
        # Get value column
        value_col = get_value_column(df)
        if value_col is None:
            print(f"  ✗ {file}: No value column found")
            continue
        
        # Log10
        df = log10_transform(df, value_col)
        log_rows = len(df)
        
        # Z-score
        df, mean, std = zscore_normalize(df, value_col)
        
        source_dfs.append(df)
        stats.append((file, mean, std))
        
        print(f"  ✓ {file}: {original_rows} → {log_rows}")
        print(f"    Z-score ({value_col}): mean={mean:.4f}, std={std:.4f}")
    
    if source_dfs:
        combined = pd.concat(source_dfs, ignore_index=True)
        normalized_sources[prop_name] = combined
        norm_stats[prop_name] = stats
        print(f"  Combined: {len(combined)} rows\n")

# Save SOURCE
for prop_name, df in normalized_sources.items():
    output_path = os.path.join(output_dir, f"{prop_name}_source_normalized.csv")
    df.to_csv(output_path, index=False)
    print(f"✓ Saved: {output_path}")

Normalizing SOURCE data (Log10 + Z-score)...

Property: clearance
    Removing 336 non-positive values
  ✓ chembl_CL_raw_smart_deduplicated.csv: 79821 → 79485
    Z-score (value): mean=1.3787, std=0.9434
  ✓ biogen_CL_raw_smart_deduplicated.csv: 3521 → 3521
    Z-score (value): mean=0.0720, std=0.2060
  Combined: 83006 rows

Property: half_life
  ✓ tdc_half_life_smart_deduplicated.csv: 665 → 665
    Z-score (Y): mean=0.6710, std=0.5987
  Combined: 665 rows

Property: fu
    Removing 29 non-positive values
  ✓ chembl_Fu_raw_smart_deduplicated.csv: 14011 → 13982
    Z-score (value): mean=-1.3081, std=0.9860
  Combined: 13982 rows

✓ Saved: ..\data\normalized_data_v2\clearance_source_normalized.csv
✓ Saved: ..\data\normalized_data_v2\half_life_source_normalized.csv
✓ Saved: ..\data\normalized_data_v2\fu_source_normalized.csv


## Copy TARGET (Y is not normalised)

In [19]:
print("\nCopying TARGET data (no normalization on Y labels)...\n")

for prop_name in ['clearance', 'half_life', 'fu']:
    print(f"Property: {prop_name}")
    
    target_files = [
        f"{prop_name}_target_train.csv",
        f"{prop_name}_target_val.csv",
        f"{prop_name}_target_test.csv"
    ]
    
    target_dfs = []
    
    for file in target_files:
        file_path = os.path.join(pksmarts_dir, file)
        if not os.path.exists(file_path):
            print(f"  ⚠️ {file} not found")
            continue
        
        df = pd.read_csv(file_path)
        
        # ⚠️ The value column from SOURCE must also be log10-transformed!
        if 'value' in df.columns:
            # Values from SOURCE get Log10 + Z-score
            value_col = 'value'
            # Remove negative values
            df = df[df[value_col] > 0].reset_index(drop=True)
            # Log10
            df[value_col] = np.log10(df[value_col])
            # Z-score (using SOURCE statistics or TARGET's own statistics)
            mean = df[value_col].mean()
            std = df[value_col].std()
            df[value_col] = (df[value_col] - mean) / std if std > 0 else 0
        
        # Leave Y as is (binary label: 0 or 1)
        target_dfs.append(df)
        print(f"  ✓ {file}: {len(df)} rows")
        if 'Y' in df.columns:
            print(f"    Y: unique={df['Y'].unique()}, mean={df['Y'].mean():.4f}")
    
    if target_dfs:
        combined = pd.concat(target_dfs, ignore_index=True)
        output_path = os.path.join(output_dir, f"{prop_name}_target_combined.csv")
        combined.to_csv(output_path, index=False)
        print(f"  Combined: {len(combined)} rows\n")
        print(f"✓ Saved: {output_path}")


Copying TARGET data (no normalization on Y labels)...

Property: clearance
  ✓ clearance_target_train.csv: 616 rows
    Y: unique=[0 1], mean=0.4659
  ✓ clearance_target_val.csv: 154 rows
    Y: unique=[1 0], mean=0.5325
  ✓ clearance_target_test.csv: 257 rows
    Y: unique=[0 1], mean=0.4864
  Combined: 1027 rows

✓ Saved: ..\data\normalized_data_v2\clearance_target_combined.csv
Property: half_life
  ✓ half_life_target_train.csv: 698 rows
    Y: unique=[1 0], mean=0.4713
  ✓ half_life_target_val.csv: 175 rows
    Y: unique=[1 0], mean=0.6286
  ✓ half_life_target_test.csv: 291 rows
    Y: unique=[1 0], mean=0.4605
  Combined: 1164 rows

✓ Saved: ..\data\normalized_data_v2\half_life_target_combined.csv
Property: fu
  ✓ fu_target_train.csv: 212 rows
    Y: unique=[0 1], mean=0.4434
  ✓ fu_target_val.csv: 53 rows
    Y: unique=[1 0], mean=0.2642
  ✓ fu_target_test.csv: 88 rows
    Y: unique=[0 1], mean=0.4318
  Combined: 353 rows

✓ Saved: ..\data\normalized_data_v2\fu_target_combined.cs

## Validation

In [20]:
print("\nVerification:")
print("="*100)

for prop_name in ['clearance', 'half_life', 'fu']:
    src_path = os.path.join(output_dir, f"{prop_name}_source_normalized.csv")
    tgt_path = os.path.join(output_dir, f"{prop_name}_target_combined.csv")
    
    print(f"\n{prop_name.upper()}:")
    
    if os.path.exists(src_path):
        src_df = pd.read_csv(src_path)
        value_col = get_value_column(src_df)
        print(f"  SOURCE: {len(src_df)} rows")
        if value_col:
            print(f"    {value_col}: mean={src_df[value_col].mean():.6f}, std={src_df[value_col].std():.6f}")
    
    if os.path.exists(tgt_path):
        tgt_df = pd.read_csv(tgt_path)
        print(f"  TARGET: {len(tgt_df)} rows")
        if 'value' in tgt_df.columns:
            print(f"    value: mean={tgt_df['value'].mean():.6f}, std={tgt_df['value'].std():.6f}")
        if 'Y' in tgt_df.columns:
            print(f"    Y (labels): unique={sorted(tgt_df['Y'].unique())}, mean={tgt_df['Y'].mean():.4f}")


Verification:

CLEARANCE:
  SOURCE: 83006 rows
    value: mean=-0.000000, std=0.999994
  TARGET: 1027 rows
    value: mean=0.000000, std=0.999025
    Y (labels): unique=[np.int64(0), np.int64(1)], mean=0.4810

HALF_LIFE:
  SOURCE: 665 rows
    Y: mean=-0.000000, std=1.000000
  TARGET: 1164 rows
    value: mean=-0.000000, std=0.999140
    Y (labels): unique=[np.int64(0), np.int64(1)], mean=0.4923

FU:
  SOURCE: 13982 rows
    value: mean=-0.000000, std=1.000000
  TARGET: 353 rows
    value: mean=-0.000000, std=0.997155
    Y (labels): unique=[np.int64(0), np.int64(1)], mean=0.4136


~\AppData\Local\Temp\ipykernel_25892\3529653936.py:11: DtypeWarning: Columns (0: Internal ID, 1: Vendor ID, 2: CollectionName) have mixed types. Specify dtype option on import or set low_memory=False.
  src_df = pd.read_csv(src_path)
